In [13]:
import os
import pandas as pd

from odin import PostgresWrapper

from rockyclickup.utils import response_to_dataframe
from rockyclickup.wrapper import Session

from rockyclickup.models import DataFile


In [11]:

rcu = Session()

In [2]:
ROCKYDB_HOST = os.getenv("ROCKYDB_HOST")
ROCKYDB_USR = os.getenv("ROCKYDB_USR")
ROCKYDB_PWD = os.getenv("ROCKYDB_PWD")
ROCKYDB_PSK = os.getenv("ROCKYDB_PSK")
ROCKYDB_SALT = os.getenv("ROCKYDB_SALT")
COREDB_GCLOUD_CREDS_PATH = os.getenv("COREDB_GCLOUD_CREDS_PATH")


coredb = PostgresWrapper(
    google_creds_path=COREDB_GCLOUD_CREDS_PATH,
    host=ROCKYDB_HOST,
    db_name="core",
    user=ROCKYDB_USR,
    password=ROCKYDB_PWD,
    ip_type="public"
)

coredb.test_connection()

Connected to 'core' via 34.106.196.176:5432


True

In [3]:
DATAINBOX_DB_CONNECTION_NAME = os.getenv("DATAINBOX_DB_CONNECTION_NAME")
DATAINBOX_DB_NAME = os.getenv("DATAINBOX_DB_NAME")
DATAINBOX_USER = os.getenv("DATAINBOX_USER")
DATAINBOX_PASSWORD = os.getenv("DATAINBOX_PASSWORD")
DATAINBOX_DB_IP_TYPE = os.getenv("DATAINBOX_DB_IP_TYPE")

inboxdb = PostgresWrapper(
    db_name=DATAINBOX_DB_NAME,
    password=DATAINBOX_PASSWORD,
    instance_connection_name=DATAINBOX_DB_CONNECTION_NAME,
    user=DATAINBOX_USER
)

inboxdb.test_connection()

Connected to 'postgres' via rmr-cloud-services:us-west1:datainbox


True

In [6]:
all_inboxdb_items = inboxdb.get_all_from_table("datafiles")

inboxdb_df = pd.DataFrame(all_inboxdb_items)

inboxdb_df = inboxdb_df.sort_values("date_created", ascending=False)

inboxdb_df.to_pickle("lists/gateway_db.pkl")


In [7]:
import datetime as dt

all_coredb_items = coredb.query(
    """
    SELECT * FROM datafile
    WHERE date_created >= %s
    """,
    [dt.datetime(2026, 8, 12)]
)

coredb_df = pd.DataFrame(all_coredb_items)

coredb_df = coredb_df.sort_values("date_created", ascending=False)

coredb_df.to_pickle("lists/rmrcloud_db.pkl")


In [ ]:
gateway_inbox = rcu.get_full_list(list_id=901114339591)

len(gateway_inbox)

gateway_cu_df = response_to_dataframe(gateway_inbox)

gateway_cu_df["date_created"] = pd.to_datetime(gateway_cu_df["date_created"], utc=True)

gateway_cu_df = gateway_cu_df.sort_values("date_created", ascending=False)

gateway_cu_df.to_pickle("lists/gateway_cu.pkl")


Field not in config.db:	2947ad1a-9b12-4800-82c4-138325278c8c	`debug_data_id`
Field not in config.db:	032df49e-6d2b-47ee-b40a-4794650c488a	`debug_assignee_id`


In [14]:

datainbox_items = rcu.get_full_list(DataFile.list_id)

datainbox_df = response_to_dataframe(datainbox_items)

datainbox_df['date_created'] = pd.to_datetime(datainbox_df["date_created"], utc=True)

datainbox_df = datainbox_df.sort_values("date_created", ascending=False)

datainbox_df.to_pickle("lists/rmrcloud_cu.pkl")